# Redrob Challenge — Initial Exploration

first pass at understanding the data

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# load sample to start with
with open('../data/sample_candidates.json') as f:
    sample = json.load(f)

print(f'Loaded {len(sample)} candidates')
print('Keys:', list(sample[0].keys()))

In [ ]:
# basic profile stats
yoe = [c['profile']['years_of_experience'] for c in sample]
titles = [c['profile']['current_title'] for c in sample]
industries = [c['profile']['current_industry'] for c in sample]

print('YOE stats:')
print(f'  mean={np.mean(yoe):.1f}, min={min(yoe)}, max={max(yoe)}')
print()
print('Top 10 titles:')
for t, cnt in Counter(titles).most_common(10):
    print(f'  {cnt}x {t}')

In [ ]:
# look at redrob signals distribution
open_to_work = [c['redrob_signals']['open_to_work_flag'] for c in sample]
response_rates = [c['redrob_signals']['recruiter_response_rate'] for c in sample]
github_scores = [c['redrob_signals']['github_activity_score'] for c in sample]

print(f"Open to work: {sum(open_to_work)}/{len(open_to_work)} ({100*sum(open_to_work)/len(open_to_work):.0f}%)")
print(f"Response rate: mean={np.mean(response_rates):.2f}")
print(f"GitHub score: {sum(1 for g in github_scores if g == -1)} have no GitHub")
print(f"GitHub score (those with it): mean={np.mean([g for g in github_scores if g >= 0]):.1f}")

In [ ]:
# check what skills appear most often
all_skills = []
for c in sample:
    all_skills.extend([s['name'] for s in c['skills']])

print('Most common skills in sample:')
for s, cnt in Counter(all_skills).most_common(20):
    print(f'  {cnt}x {s}')

In [ ]:
# yoe distribution plot
plt.figure(figsize=(8, 4))
plt.hist(yoe, bins=12, color='teal', edgecolor='white')
plt.xlabel('Years of Experience')
plt.ylabel('Count')
plt.title('Experience Distribution (sample)')
plt.axvline(5, color='navy', linestyle='--', label='5yr (JD min)')
plt.axvline(9, color='navy', linestyle='--', label='9yr (JD max)')
plt.legend()
plt.tight_layout()
plt.savefig('../output/yoe_distribution.png', dpi=100)
plt.show()

In [ ]:
# education tiers
tiers = []
for c in sample:
    for e in c.get('education', []):
        tiers.append(e.get('tier', 'unknown'))

print('Education tiers in sample:')
for t, cnt in Counter(tiers).most_common():
    print(f'  {t}: {cnt}')

In [ ]:
# look at a specific candidate in detail to understand the data
c = sample[0]
print('=== Candidate Detail ===')
print('ID:', c['candidate_id'])
print('Title:', c['profile']['current_title'])
print('Headline:', c['profile']['headline'])
print('YOE:', c['profile']['years_of_experience'])
print()
print('Skills:')
for s in c['skills']:
    print(f"  {s['name']} ({s['proficiency']}) - {s.get('duration_months', '?')} months")
print()
print('Redrob signals:')
sig = c['redrob_signals']
print(f"  open_to_work: {sig['open_to_work_flag']}")
print(f"  recruiter_response_rate: {sig['recruiter_response_rate']}")
print(f"  last_active_date: {sig['last_active_date']}")
print(f"  github_activity_score: {sig['github_activity_score']}")
print(f"  notice_period_days: {sig['notice_period_days']}")

In [ ]:
# quick baseline: keyword matching
# just checking which sample candidates have the most JD keyword hits

must_have = [
    'embeddings', 'sentence-transformers', 'vector', 'faiss', 'pinecone',
    'python', 'ranking', 'retrieval', 'nlp', 'llm', 'information retrieval',
    'ndcg', 'semantic search', 'hybrid search', 'transformer'
]

def count_hits(candidate, keywords):
    text = json.dumps(candidate).lower()
    return sum(1 for kw in keywords if kw in text)

hits = [(c['candidate_id'], c['profile']['current_title'], count_hits(c, must_have)) for c in sample]
hits.sort(key=lambda x: -x[2])

print('Top candidates by keyword hit count:')
for cid, title, h in hits[:10]:
    print(f'  {cid} | {title:30s} | {h} hits')

In [ ]:
# try TF-IDF approach - treating each candidate as a document
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# build corpus
def candidate_to_text(c):
    parts = []
    p = c['profile']
    parts.append(p.get('headline', ''))
    parts.append(p.get('summary', ''))
    parts.append(p.get('current_title', ''))
    for s in c['skills']:
        parts.append(s['name'])
    for job in c.get('career_history', [])[:2]:
        parts.append(job.get('description', ''))
    return ' '.join(parts)

jd_text = """
Senior AI Engineer embeddings retrieval vector database sentence transformers Python
ranking evaluation NDCG MRR semantic search hybrid search NLP information retrieval
LLM fine-tuning learning to rank FAISS Pinecone Weaviate Qdrant Elasticsearch
"""

texts = [candidate_to_text(c) for c in sample]
corpus = [jd_text] + texts

tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
matrix = tfidf.fit_transform(corpus)

sims = cosine_similarity(matrix[0:1], matrix[1:])[0]

# show top by tfidf
top_idx = np.argsort(-sims)[:5]
print('Top 5 by TF-IDF:')
for i in top_idx:
    print(f"  {sample[i]['candidate_id']} | {sample[i]['profile']['current_title']:30s} | sim={sims[i]:.3f}")

## Observations after day 1-2 exploration

1. Sample has 50 candidates, quite diverse titles (many are not AI roles)
2. YOE range is wide - many candidates outside 5-9 year target
3. About 60% are open to work
4. Around 30-40% have no GitHub linked (score = -1)
5. TF-IDF gives reasonable baseline but misses semantic meaning
6. Need to combine: skill match + semantic + behavioral signals
7. Behavioral signals are key - active candidates much more valuable

Next steps: build proper feature_engineering.py and try sentence-transformers